In [1]:
import pandas as pd
import numpy as np
import chromadb
import re
from IPython.display import display

In [2]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_collection(
    name="insurance_documents"
)

print("Connected to ChromaDB!")
print("Total chunks:", collection.count())

Connected to ChromaDB!
Total chunks: 872


In [3]:
questions = [
    "What are the expenses covered under inpatient hospitalization?",
    "What are day care treatments? Give some examples.",
    "What is the pre-hospitalization and post-hospitalization period?",
    "What is domiciliary hospitalization?",
    "What is Automatic Recharge?"
]

for i, question in enumerate(questions, start=1):
    print(f"Q{i}: {question}")

Q1: What are the expenses covered under inpatient hospitalization?
Q2: What are day care treatments? Give some examples.
Q3: What is the pre-hospitalization and post-hospitalization period?
Q4: What is domiciliary hospitalization?
Q5: What is Automatic Recharge?


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [5]:
def dense_retrieve(question, k=10):

    query_embedding = embedding_model.embed_query(question)

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for i, chunk in enumerate(result["documents"][0]):
        retrieved.append({
            "chunk_id": result["ids"][0][i],
            "chunk": chunk,
            "metadata": result["metadatas"][0][i],
            "distance": result["distances"][0][i],
            "rank": i + 1
        })

    return retrieved

In [6]:
dense_results = []

for qid, question in enumerate(questions, start=1):

    retrieved_chunks = dense_retrieve(question, k=10)

    for item in retrieved_chunks:
        dense_results.append({
            "question_id": qid,
            "question": question,
            **item
        })

dense_df = pd.DataFrame(dense_results)

display(
    dense_df[
        ["question_id", "question", "rank", "chunk_id", "chunk", "distance"]
    ]
)

,question_id,question,rank,chunk_id,chunk,distance
0,1,What are the expenses covered under inpatient ...,1,chunk_577,It all adds up - A trip to a hospital involves...,0.484362
1,1,What are the expenses covered under inpatient ...,2,chunk_506,Expenses: (i) Pre-Hospitalization Medical Expe...,0.498260
2,1,What are the expenses covered under inpatient ...,3,chunk_502,"care due to Illness or Injury , which should b...",0.595449
3,1,What are the expenses covered under inpatient ...,4,chunk_851,for expenses incurred under Benefit 'In-patien...,0.615453
4,1,What are the expenses covered under inpatient ...,5,chunk_705,Sum Insured (SI) – on annual basis (in Rs.) 3 ...,0.626415
5,1,What are the expenses covered under inpatient ...,6,chunk_477,L) Pre-Hospitalization Medical Expenses and Po...,0.630231
6,1,What are the expenses covered under inpatient ...,7,chunk_25,We also pay for your medical expenses if you u...,0.646136
7,1,What are the expenses covered under inpatient ...,8,chunk_0,Policy Usage Guide In-patient Hospitalization ...,0.655014
8,1,What are the expenses covered under inpatient ...,9,chunk_75,- perhaps Your state of health is such that Yo...,0.659194
9,1,What are the expenses covered under inpatient ...,10,chunk_507,"Hospitalization, provided that We shall not be...",0.664582


In [7]:
ground_truth = {
    1: set(),  # Q1 ke saare verified relevant chunk IDs
    2: set(),  # Q2
    3: set(),  # Q3
    4: set(),  # Q4
    5: set()   # Q5
}

In [8]:
def calculate_recall_at_k(results_df, ground_truth, k):

    rows = []

    for qid, group in results_df.groupby("question_id"):

        relevant_ids = ground_truth.get(qid, set())

        top_k_ids = set(
            group.sort_values("rank")
            .head(k)["chunk_id"]
        )

        if len(relevant_ids) == 0:
            recall = np.nan
            relevant_retrieved = 0
        else:
            relevant_retrieved = len(top_k_ids & relevant_ids)
            recall = relevant_retrieved / len(relevant_ids)

        rows.append({
            "question_id": qid,
            "question": group["question"].iloc[0],
            "K": k,
            "relevant_retrieved": relevant_retrieved,
            "total_relevant": len(relevant_ids),
            "Recall": recall
        })

    return pd.DataFrame(rows)

In [9]:
recall_tables = []

for k in [1, 3, 5, 10]:
    recall_tables.append(
        calculate_recall_at_k(dense_df, ground_truth, k)
    )

dense_recall_df = pd.concat(
    recall_tables,
    ignore_index=True
)

display(dense_recall_df)

,question_id,question,K,relevant_retrieved,total_relevant,Recall
0,1,What are the expenses covered under inpatient ...,1,0,0,NaN
1,2,What are day care treatments? Give some examples.,1,0,0,NaN
2,3,What is the pre-hospitalization and post-hospi...,1,0,0,NaN
3,4,What is domiciliary hospitalization?,1,0,0,NaN
4,5,What is Automatic Recharge?,1,0,0,NaN
5,1,What are the expenses covered under inpatient ...,3,0,0,NaN
6,2,What are day care treatments? Give some examples.,3,0,0,NaN
7,3,What is the pre-hospitalization and post-hospi...,3,0,0,NaN
8,4,What is domiciliary hospitalization?,3,0,0,NaN
9,5,What is Automatic Recharge?,3,0,0,NaN


In [10]:
for qid, relevant_ids in ground_truth.items():
    print(f"Q{qid}: {len(relevant_ids)} relevant chunks")

Q1: 0 relevant chunks
Q2: 0 relevant chunks
Q3: 0 relevant chunks
Q4: 0 relevant chunks
Q5: 0 relevant chunks


In [11]:
for qid in sorted(dense_df["question_id"].unique()):
    print(f"\n{'='*80}")
    print(f"QUESTION {qid}:")
    print(
        dense_df.loc[dense_df["question_id"] == qid, "question"].iloc[0]
    )

    display(
        dense_df.loc[
            dense_df["question_id"] == qid,
            ["rank", "chunk_id", "chunk"]
        ]
    )


QUESTION 1:
What are the expenses covered under inpatient hospitalization?


,rank,chunk_id,chunk
0,1,chunk_577,It all adds up - A trip to a hospital involves...
1,2,chunk_506,Expenses: (i) Pre-Hospitalization Medical Expe...
2,3,chunk_502,"care due to Illness or Injury , which should b..."
3,4,chunk_851,for expenses incurred under Benefit 'In-patien...
4,5,chunk_705,Sum Insured (SI) – on annual basis (in Rs.) 3 ...
5,6,chunk_477,L) Pre-Hospitalization Medical Expenses and Po...
6,7,chunk_25,We also pay for your medical expenses if you u...
7,8,chunk_0,Policy Usage Guide In-patient Hospitalization ...
8,9,chunk_75,- perhaps Your state of health is such that Yo...
9,10,chunk_507,"Hospitalization, provided that We shall not be..."



QUESTION 2:
What are day care treatments? Give some examples.


,rank,chunk_id,chunk
10,1,chunk_503,which forms a part of Hospitalization. (ii) Da...
11,2,chunk_163,for personal care such as help with activities...
12,3,chunk_850,"such as room charges, nursing expenses, intens..."
13,4,chunk_379,either at home or in a nursing facility for pe...
14,5,chunk_0,Policy Usage Guide In-patient Hospitalization ...
15,6,chunk_502,"care due to Illness or Injury , which should b..."
16,7,chunk_68,"charges, etc. Please refer to the Schedule of ..."
17,8,chunk_25,We also pay for your medical expenses if you u...
18,9,chunk_24,"for in-patient care, for a minimum period of 2..."
19,10,chunk_384,"nature cure clinics, spas or similar establish..."



QUESTION 3:
What is the pre-hospitalization and post-hospitalization period?


,rank,chunk_id,chunk
20,1,chunk_72,to a Hospital for treatment of any One Illness...
21,2,chunk_507,"Hospitalization, provided that We shall not be..."
22,3,chunk_212,"10 days Rs.15,000 if Hospitalization exceeds 1..."
23,4,chunk_73,immediately before Insured Person’s Hospitaliz...
24,5,chunk_121,within 15 days from the date of discharge from...
25,6,chunk_27,POST-HOSPITALISATION Back home and till you ar...
26,7,chunk_851,for expenses incurred under Benefit 'In-patien...
27,8,chunk_513,to be moved to a Hospital or a Hospital room m...
28,9,chunk_30,treatment for a period exceeding 3 consecutive...
29,10,chunk_578,for a maximum of 15 days of Hospitalization du...



QUESTION 4:
What is domiciliary hospitalization?


,rank,chunk_id,chunk
30,1,chunk_29,"and treatment at a hospital), hospitalisation ..."
31,2,chunk_512,and other applicable laws and rules. ‘Pre Hosp...
32,3,chunk_513,to be moved to a Hospital or a Hospital room m...
33,4,chunk_28,COVER Getting to the hospital - It is our utmo...
34,5,chunk_74,3 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 5....
35,6,chunk_157,hospitals (iii) Any condition caused by or ass...
36,7,chunk_75,- perhaps Your state of health is such that Yo...
37,8,chunk_163,for personal care such as help with activities...
38,9,chunk_503,which forms a part of Hospitalization. (ii) Da...
39,10,chunk_610,and caesarean sections incurred during hospita...



QUESTION 5:
What is Automatic Recharge?


,rank,chunk_id,chunk
40,1,chunk_522,considered while calculating ‘Unlimited Automa...
41,2,chunk_2,treatment at home so long as it involves medic...
42,3,chunk_855,on an ambulance service offered by the hospita...
43,4,chunk_521,"cover, we reinstate the entire base sum insure..."
44,5,chunk_529,5 CARE PLUS - UIN: CHIHLIP 22047 V 012122 with...
45,6,chunk_31,"these expenses, we pay ‘Consumable Allowance’ ..."
46,7,chunk_77,of Dysenteries including Gastro-enteritis; v. ...
47,8,chunk_171,not limited to T.V. (wherever specifically cha...
48,9,chunk_172,"charges, service charge, surcharge, night char..."
49,10,chunk_202,Recharge of Sum Insured N.A. 100% of original ...


In [12]:
candidate_ids = [
    "chunk_506", "chunk_502", "chunk_851", "chunk_25", "chunk_0", "chunk_507",
    "chunk_503", "chunk_850", "chunk_68", "chunk_24",
    "chunk_72", "chunk_73", "chunk_121", "chunk_27",
    "chunk_29", "chunk_512", "chunk_513", "chunk_28", "chunk_75",
    "chunk_522", "chunk_521", "chunk_529", "chunk_202"
]

for chunk_id in candidate_ids:
    result = collection.get(
        ids=[chunk_id],
        include=["documents", "metadatas"]
    )

    if result["documents"]:
        print("\n" + "=" * 80)
        print("CHUNK ID:", chunk_id)
        print(result["documents"][0])


CHUNK ID: chunk_506
Expenses: (i) Pre-Hospitalization Medical Expenses: Examination, tests and medication - Sometimes the procedures that finally lead You to hospital, such as Investigative tests, Consultation Fees and medication, can be quite financially draining. We cover the medically necessary expenses (as per specified amount/limit) incurred by You for a period of 60 days immediately before the date of Your admissible Hospitalization, provided that We shall not be liable to make payment for any Pre-hospitalization

CHUNK ID: chunk_502
care due to Illness or Injury , which should be Medically Necessary, for a minimum period of 24 consecutive hours, We will pay for the medical expenses, through Cashless or Reimbursement Facility maximum up to Sum Insured, incurred by You at the hospital - from room charges, nursing expenses and intensive care unit charges to Surgeon’s fee, Doctor’s fee, Anesthesia, blood, oxygen, Operation theater charges which forms a part of Hospitalization. (ii)

In [13]:
ground_truth = {
    1: {"verified_Q1_chunk_id", "verified_Q1_chunk_id_2"},
    2: {"verified_Q2_chunk_id", "verified_Q2_chunk_id_2"},
    3: {"verified_Q3_chunk_id", "verified_Q3_chunk_id_2"},
    4: {"verified_Q4_chunk_id", "verified_Q4_chunk_id_2"},
    5: {"verified_Q5_chunk_id", "verified_Q5_chunk_id_2"}
}

In [14]:
all_chunk_ids = set(
    collection.get(include=[])["ids"]
)

for qid, relevant_ids in ground_truth.items():
    missing_ids = relevant_ids - all_chunk_ids

    print(f"Q{qid}:")
    print("Relevant IDs entered:", len(relevant_ids))
    print("IDs not found in collection:", missing_ids)

Q1:
Relevant IDs entered: 2
IDs not found in collection: {'verified_Q1_chunk_id_2', 'verified_Q1_chunk_id'}
Q2:
Relevant IDs entered: 2
IDs not found in collection: {'verified_Q2_chunk_id', 'verified_Q2_chunk_id_2'}
Q3:
Relevant IDs entered: 2
IDs not found in collection: {'verified_Q3_chunk_id_2', 'verified_Q3_chunk_id'}
Q4:
Relevant IDs entered: 2
IDs not found in collection: {'verified_Q4_chunk_id_2', 'verified_Q4_chunk_id'}
Q5:
Relevant IDs entered: 2
IDs not found in collection: {'verified_Q5_chunk_id', 'verified_Q5_chunk_id_2'}


In [15]:
for qid, group in dense_df.groupby("question_id"):

    relevant_ids = ground_truth.get(qid, set())

    top5_ids = set(
        group.sort_values("rank").head(5)["chunk_id"]
    )

    print(f"\nQ{qid}")
    print("Ground-truth IDs:", relevant_ids)
    print("Top-5 retrieved IDs:", top5_ids)
    print("Matching IDs:", top5_ids & relevant_ids)
    print("Total relevant IDs:", len(relevant_ids))


Q1
Ground-truth IDs: {'verified_Q1_chunk_id_2', 'verified_Q1_chunk_id'}
Top-5 retrieved IDs: {'chunk_705', 'chunk_506', 'chunk_502', 'chunk_577', 'chunk_851'}
Matching IDs: set()
Total relevant IDs: 2

Q2
Ground-truth IDs: {'verified_Q2_chunk_id_2', 'verified_Q2_chunk_id'}
Top-5 retrieved IDs: {'chunk_0', 'chunk_850', 'chunk_503', 'chunk_379', 'chunk_163'}
Matching IDs: set()
Total relevant IDs: 2

Q3
Ground-truth IDs: {'verified_Q3_chunk_id_2', 'verified_Q3_chunk_id'}
Top-5 retrieved IDs: {'chunk_507', 'chunk_73', 'chunk_121', 'chunk_212', 'chunk_72'}
Matching IDs: set()
Total relevant IDs: 2

Q4
Ground-truth IDs: {'verified_Q4_chunk_id_2', 'verified_Q4_chunk_id'}
Top-5 retrieved IDs: {'chunk_28', 'chunk_29', 'chunk_74', 'chunk_512', 'chunk_513'}
Matching IDs: set()
Total relevant IDs: 2

Q5
Ground-truth IDs: {'verified_Q5_chunk_id', 'verified_Q5_chunk_id_2'}
Top-5 retrieved IDs: {'chunk_2', 'chunk_522', 'chunk_855', 'chunk_529', 'chunk_521'}
Matching IDs: set()
Total relevant IDs: 2

In [16]:
ground_truth = {
    1: {"chunk_502", "chunk_506"},  # Q1: retain if already verified
    2: {"chunk_503", "chunk_850", "chunk_68", "chunk_502"},
    3: {"chunk_72", "chunk_73", "chunk_27"},
    4: {"chunk_29", "chunk_512", "chunk_513", "chunk_28", "chunk_75"},
    5: {"chunk_522", "chunk_521", "chunk_202"}
}

In [17]:
for qid, group in dense_df.groupby("question_id"):
    relevant_ids = ground_truth.get(qid, set())
    top5_ids = set(group.sort_values("rank").head(5)["chunk_id"])

    print(
        f"Q{qid}: matching IDs =",
        top5_ids & relevant_ids
    )

Q1: matching IDs = {'chunk_506', 'chunk_502'}
Q2: matching IDs = {'chunk_850', 'chunk_503'}
Q3: matching IDs = {'chunk_73', 'chunk_72'}
Q4: matching IDs = {'chunk_512', 'chunk_29', 'chunk_513', 'chunk_28'}
Q5: matching IDs = {'chunk_522', 'chunk_521'}


In [18]:
candidate_ids = {
    2: ["chunk_503", "chunk_850", "chunk_68", "chunk_502", "chunk_24"],
    3: ["chunk_72", "chunk_73", "chunk_121", "chunk_27", "chunk_507"],
    4: ["chunk_29", "chunk_512", "chunk_513", "chunk_28", "chunk_75"],
    5: ["chunk_522", "chunk_521", "chunk_529", "chunk_202"]
}

for q_id, ids in candidate_ids.items():
    print("\n" + "#" * 90)
    print(f"QUESTION Q{q_id}")

    for chunk_id in ids:
        result = collection.get(
            ids=[chunk_id],
            include=["documents", "metadatas"]
        )

        if result["documents"]:
            print("\n" + "=" * 80)
            print("CHUNK ID:", chunk_id)
            print("TEXT:", result["documents"][0])
        else:
            print(f"\nChunk not found: {chunk_id}")


##########################################################################################
QUESTION Q2

CHUNK ID: chunk_503
TEXT: which forms a part of Hospitalization. (ii) Day Care Treatment: Hospitalization involving less than 24 hours – Some surgeries doesn’t require or need not necessarily require Hospitalization Stay for minimum 24 Hours. It may be for your convenience or it may happen that the surgery underwent is minor or of intermediate complexity. We will pay through Cashless or Reimbursement Facility for all such day care treatments, maximum up to Sum Insured. (iii) Advance Technology Methods: We will indemnify you for

CHUNK ID: chunk_850
TEXT: such as room charges, nursing expenses, intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theatre charges, etc. Day Care Treatment Indemnifies up to the Sum Insured for the medical expenses incurred during treatments that require the Insured Person to be hospitalized for less than 24 hour

In [19]:
for q_id, relevant_ids in ground_truth.items():
    valid_ids = set(collection.get(ids=list(relevant_ids))["ids"])
    missing_ids = relevant_ids - valid_ids

    print(f"Q{q_id}:")
    print("Relevant IDs:", relevant_ids)
    print("Missing IDs:", missing_ids)
    print()

Q1:
Relevant IDs: {'chunk_506', 'chunk_502'}
Missing IDs: set()

Q2:
Relevant IDs: {'chunk_68', 'chunk_850', 'chunk_502', 'chunk_503'}
Missing IDs: set()

Q3:
Relevant IDs: {'chunk_27', 'chunk_73', 'chunk_72'}
Missing IDs: set()

Q4:
Relevant IDs: {'chunk_75', 'chunk_29', 'chunk_512', 'chunk_513', 'chunk_28'}
Missing IDs: set()

Q5:
Relevant IDs: {'chunk_202', 'chunk_522', 'chunk_521'}
Missing IDs: set()



In [20]:
def calculate_recall(retrieved_ids, relevant_ids):
    if not relevant_ids:
        return None
    return len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)


for q_id in range(1, 6):
    relevant_ids = ground_truth[q_id]

    print(f"\nQ{q_id}")
    for k in [1, 3, 5, 10]:
        retrieved_ids = (
            dense_df[dense_df["question_id"] == q_id]
            .sort_values("rank")
            .head(k)["chunk_id"]
            .tolist()
        )

        recall = calculate_recall(retrieved_ids, relevant_ids)
        print(f"Recall@{k}: {recall:.2f}" if recall is not None
              else f"Recall@{k}: N/A — no relevant IDs labeled")


Q1
Recall@1: 0.00
Recall@3: 1.00
Recall@5: 1.00
Recall@10: 1.00

Q2
Recall@1: 0.25
Recall@3: 0.50
Recall@5: 0.50
Recall@10: 1.00

Q3
Recall@1: 0.33
Recall@3: 0.33
Recall@5: 0.67
Recall@10: 1.00

Q4
Recall@1: 0.20
Recall@3: 0.60
Recall@5: 0.80
Recall@10: 1.00

Q5
Recall@1: 0.33
Recall@3: 0.33
Recall@5: 0.67
Recall@10: 1.00


In [21]:
%pip install rank-bm25

In [22]:
import re
from rank_bm25 import BM25Okapi

# Get all chunks from ChromaDB
all_data = collection.get(
    include=["documents", "metadatas"]
)

chunk_ids = all_data["ids"]
documents = all_data["documents"]
metadatas = all_data["metadatas"]

print("Total chunks:", len(documents))

Total chunks: 872


In [23]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


tokenized_documents = [
    tokenize(doc) for doc in documents
]

bm25 = BM25Okapi(tokenized_documents)


def bm25_retrieve(question, k=10):
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:k]

    retrieved = []

    for rank, idx in enumerate(top_indices, start=1):
        retrieved.append({
            "rank": rank,
            "chunk_id": chunk_ids[idx],
            "chunk": documents[idx],
            "metadata": metadatas[idx],
            "score": float(scores[idx])
        })

    return retrieved

In [25]:
bm25_rows = []

for q_id, question in enumerate(questions, start=1):
    results = bm25_retrieve(question, k=10)

    for item in results:
        bm25_rows.append({
            "question_id": q_id,
            "question": question,
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "chunk": item["chunk"],
            "metadata": item["metadata"],
            "score": item["score"]
        })

import pandas as pd

bm25_df = pd.DataFrame(bm25_rows)

display(bm25_df)

,question_id,question,rank,chunk_id,chunk,metadata,score
0,1,What are the expenses covered under inpatient ...,1,chunk_851,for expenses incurred under Benefit 'In-patien...,"{'page': 1, 'source': 'care-supreme----policy-...",15.181555
1,1,What are the expenses covered under inpatient ...,2,chunk_574,is not available in case your age is greater t...,"{'page': 10, 'source': 'care-plus--(health-ins...",14.297101
2,1,What are the expenses covered under inpatient ...,3,chunk_575,"under this Benefit. However, these expenses ar...",{'document_type': 'Prospectus / Sales Literatu...,14.018149
3,1,What are the expenses covered under inpatient ...,4,chunk_57,caesarean sections incurred during hospitaliza...,{'source': 'care-freedom-(health-insurance-pro...,13.376425
4,1,What are the expenses covered under inpatient ...,5,chunk_13,A mandatory Co-Payment of 10% per claim is app...,"{'page': 3, 'document_type': 'Policy Usage Gui...",12.561187
5,1,What are the expenses covered under inpatient ...,6,chunk_859,WHAT IS NOT COVERED? Any hospital admission pr...,"{'document_type': 'Policy Usage Guide', 'page'...",11.875643
6,1,What are the expenses covered under inpatient ...,7,chunk_854,at home which had actually merited domiciliary...,"{'document_type': 'Policy Usage Guide', 'page'...",11.581680
7,1,What are the expenses covered under inpatient ...,8,chunk_630,identification proof as stated by us. (iii) Ap...,{'document_type': 'Prospectus / Sales Literatu...,11.503347
8,1,What are the expenses covered under inpatient ...,9,chunk_631,Medical Expenses are covered under this Policy...,"{'page': 16, 'document_type': 'Prospectus / Sa...",11.148509
9,1,What are the expenses covered under inpatient ...,10,chunk_522,considered while calculating ‘Unlimited Automa...,"{'page': 4, 'source': 'care-plus--(health-insu...",10.962530


In [26]:
for q_id in range(1, 6):
    relevant_ids = ground_truth[q_id]

    print(f"\nQ{q_id}")

    for k in [1, 3, 5, 10]:
        retrieved_ids = (
            bm25_df[bm25_df["question_id"] == q_id]
            .sort_values("rank")
            .head(k)["chunk_id"]
            .tolist()
        )

        recall = calculate_recall(retrieved_ids, relevant_ids)

        if recall is not None:
            print(f"Recall@{k}: {recall:.2f}")
        else:
            print(f"Recall@{k}: N/A")


Q1
Recall@1: 0.00
Recall@3: 0.00
Recall@5: 0.00
Recall@10: 0.00

Q2
Recall@1: 0.00
Recall@3: 0.25
Recall@5: 0.25
Recall@10: 0.50

Q3
Recall@1: 0.00
Recall@3: 0.00
Recall@5: 0.33
Recall@10: 0.33

Q4
Recall@1: 0.00
Recall@3: 0.00
Recall@5: 0.00
Recall@10: 0.40

Q5
Recall@1: 0.33
Recall@3: 0.67
Recall@5: 0.67
Recall@10: 0.67


In [27]:
import pandas as pd

dense_recall_results = {
    "Q1": [0.00, 1.00, 1.00, 1.00],
    "Q2": [0.25, 0.50, 0.50, 1.00],
    "Q3": [0.33, 0.33, 0.67, 1.00],
    "Q4": [0.20, 0.60, 0.80, 1.00],
    "Q5": [0.33, 0.33, 0.67, 1.00]
}

bm25_recall_results = {
    "Q1": [0.00, 0.00, 0.00, 0.00],
    "Q2": [0.00, 0.25, 0.25, 0.50],
    "Q3": [0.00, 0.00, 0.33, 0.33],
    "Q4": [0.00, 0.00, 0.00, 0.40],
    "Q5": [0.33, 0.67, 0.67, 0.67]
}

k_values = ["Recall@1", "Recall@3", "Recall@5", "Recall@10"]

comparison_rows = []

for method, results in [
    ("Dense", dense_recall_results),
    ("BM25", bm25_recall_results)
]:
    for i, k in enumerate(k_values):
        mean_value = sum(
            question_values[i] for question_values in results.values()
        ) / len(results)

        comparison_rows.append({
            "Method": method,
            "Metric": k,
            "Mean Recall": round(mean_value, 3)
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,Method,Metric,Mean Recall
0,Dense,Recall@1,0.222
1,Dense,Recall@3,0.552
2,Dense,Recall@5,0.728
3,Dense,Recall@10,1.000
4,BM25,Recall@1,0.066
5,BM25,Recall@3,0.184
6,BM25,Recall@5,0.250
7,BM25,Recall@10,0.380


In [28]:
def calculate_precision(retrieved_ids, relevant_ids):
    if not retrieved_ids:
        return 0.0

    return len(set(retrieved_ids) & relevant_ids) / len(retrieved_ids)


def evaluate_retriever(df, method_name):
    rows = []

    for q_id in range(1, 6):
        relevant_ids = ground_truth[q_id]

        for k in [1, 3, 5, 10]:
            retrieved_ids = (
                df[df["question_id"] == q_id]
                .sort_values("rank")
                .head(k)["chunk_id"]
                .tolist()
            )

            precision = calculate_precision(retrieved_ids, relevant_ids)
            recall = calculate_recall(retrieved_ids, relevant_ids)

            rows.append({
                "Question": f"Q{q_id}",
                "Method": method_name,
                "K": k,
                "Precision": round(precision, 3),
                "Recall": round(recall, 3)
            })

    return pd.DataFrame(rows)


dense_metrics = evaluate_retriever(dense_df, "Dense")
bm25_metrics = evaluate_retriever(bm25_df, "BM25")

metrics_df = pd.concat(
    [dense_metrics, bm25_metrics],
    ignore_index=True
)

display(metrics_df)

,Question,Method,K,Precision,Recall
0,Q1,Dense,1,0.000,0.000
1,Q1,Dense,3,0.667,1.000
2,Q1,Dense,5,0.400,1.000
3,Q1,Dense,10,0.200,1.000
4,Q2,Dense,1,1.000,0.250
5,Q2,Dense,3,0.667,0.500
6,Q2,Dense,5,0.400,0.500
7,Q2,Dense,10,0.400,1.000
8,Q3,Dense,1,1.000,0.333
9,Q3,Dense,3,0.333,0.333


In [29]:
mean_metrics = (
    metrics_df
    .groupby(["Method", "K"])[["Precision", "Recall"]]
    .mean()
    .reset_index()
    .round(3)
)

display(mean_metrics)

,Method,K,Precision,Recall
0,BM25,1,0.20,0.067
1,BM25,3,0.20,0.183
2,BM25,5,0.16,0.250
3,BM25,10,0.14,0.380
4,Dense,1,0.80,0.223
5,Dense,3,0.60,0.553
6,Dense,5,0.48,0.727
7,Dense,10,0.34,1.000


In [30]:
def hybrid_retrieve(question_id, k=10, rrf_constant=60):
    dense_results = (
        dense_df[dense_df["question_id"] == question_id]
        .sort_values("rank")
    )

    bm25_results = (
        bm25_df[bm25_df["question_id"] == question_id]
        .sort_values("rank")
    )

    fused_scores = {}
    chunk_texts = {}
    chunk_metadatas = {}

    # Add Dense Retrieval rankings
    for _, row in dense_results.iterrows():
        chunk_id = row["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + 1 / (rrf_constant + row["rank"])
        )

        chunk_texts[chunk_id] = row["chunk"]
        chunk_metadatas[chunk_id] = row["metadata"]

    # Add BM25 rankings
    for _, row in bm25_results.iterrows():
        chunk_id = row["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + 1 / (rrf_constant + row["rank"])
        )

        chunk_texts[chunk_id] = row["chunk"]
        chunk_metadatas[chunk_id] = row["metadata"]

    # Sort by fused RRF score
    ranked_chunks = sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:k]

    return [
        {
            "rank": rank,
            "chunk_id": chunk_id,
            "chunk": chunk_texts[chunk_id],
            "metadata": chunk_metadatas[chunk_id],
            "rrf_score": score
        }
        for rank, (chunk_id, score) in enumerate(ranked_chunks, start=1)
    ]

In [31]:
hybrid_rows = []

for q_id, question in enumerate(questions, start=1):
    results = hybrid_retrieve(q_id, k=10)

    for item in results:
        hybrid_rows.append({
            "question_id": q_id,
            "question": question,
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "chunk": item["chunk"],
            "metadata": item["metadata"],
            "rrf_score": item["rrf_score"]
        })

hybrid_df = pd.DataFrame(hybrid_rows)

display(hybrid_df)

,question_id,question,rank,chunk_id,chunk,metadata,rrf_score
0,1,What are the expenses covered under inpatient ...,1,chunk_851,for expenses incurred under Benefit 'In-patien...,"{'page': 1, 'source': 'care-supreme----policy-...",0.032018
1,1,What are the expenses covered under inpatient ...,2,chunk_577,It all adds up - A trip to a hospital involves...,{'document_type': 'Prospectus / Sales Literatu...,0.016393
2,1,What are the expenses covered under inpatient ...,3,chunk_506,Expenses: (i) Pre-Hospitalization Medical Expe...,"{'page': 2, 'document_type': 'Prospectus / Sal...",0.016129
3,1,What are the expenses covered under inpatient ...,4,chunk_574,is not available in case your age is greater t...,"{'page': 10, 'source': 'care-plus--(health-ins...",0.016129
4,1,What are the expenses covered under inpatient ...,5,chunk_502,"care due to Illness or Injury , which should b...",{'document_type': 'Prospectus / Sales Literatu...,0.015873
5,1,What are the expenses covered under inpatient ...,6,chunk_575,"under this Benefit. However, these expenses ar...",{'document_type': 'Prospectus / Sales Literatu...,0.015873
6,1,What are the expenses covered under inpatient ...,7,chunk_57,caesarean sections incurred during hospitaliza...,{'source': 'care-freedom-(health-insurance-pro...,0.015625
7,1,What are the expenses covered under inpatient ...,8,chunk_705,Sum Insured (SI) – on annual basis (in Rs.) 3 ...,{'source': 'care-plus--(health-insurance-produ...,0.015385
8,1,What are the expenses covered under inpatient ...,9,chunk_13,A mandatory Co-Payment of 10% per claim is app...,"{'page': 3, 'document_type': 'Policy Usage Gui...",0.015385
9,1,What are the expenses covered under inpatient ...,10,chunk_477,L) Pre-Hospitalization Medical Expenses and Po...,"{'document_type': 'Brochure', 'source': 'care-...",0.015152


In [32]:
hybrid_metrics = evaluate_retriever(hybrid_df, "Hybrid")

display(hybrid_metrics)

,Question,Method,K,Precision,Recall
0,Q1,Hybrid,1,0.000,0.000
1,Q1,Hybrid,3,0.333,0.500
2,Q1,Hybrid,5,0.400,1.000
3,Q1,Hybrid,10,0.200,1.000
4,Q2,Hybrid,1,1.000,0.250
5,Q2,Hybrid,3,0.667,0.500
6,Q2,Hybrid,5,0.400,0.500
7,Q2,Hybrid,10,0.300,0.750
8,Q3,Hybrid,1,1.000,0.333
9,Q3,Hybrid,3,0.333,0.333


In [33]:
all_metrics = pd.concat(
    [dense_metrics, bm25_metrics, hybrid_metrics],
    ignore_index=True
)

final_comparison = (
    all_metrics
    .groupby(["Method", "K"])[["Precision", "Recall"]]
    .mean()
    .reset_index()
    .round(3)
)

display(final_comparison)

,Method,K,Precision,Recall
0,BM25,1,0.20,0.067
1,BM25,3,0.20,0.183
2,BM25,5,0.16,0.250
3,BM25,10,0.14,0.380
4,Dense,1,0.80,0.223
5,Dense,3,0.60,0.553
6,Dense,5,0.48,0.727
7,Dense,10,0.34,1.000
8,Hybrid,1,0.80,0.223
9,Hybrid,3,0.60,0.520


In [34]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker model loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

C:\Users\LENOVO 4HIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO 4HIN\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker model loaded successfully!


In [35]:
def rerank_retrieved_chunks(question, retrieved_chunks, k=10):
    pairs = [
        (question, item["chunk"])
        for item in retrieved_chunks
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for item, score in zip(retrieved_chunks, scores):
        reranked.append({
            **item,
            "rerank_score": float(score)
        })

    reranked.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    for rank, item in enumerate(reranked[:k], start=1):
        item["rank"] = rank

    return reranked[:k]

In [36]:
reranked_rows = []

for q_id, question in enumerate(questions, start=1):
    candidates = (
        hybrid_df[hybrid_df["question_id"] == q_id]
        .sort_values("rank")
        .to_dict("records")
    )

    reranked_results = rerank_retrieved_chunks(
        question,
        candidates,
        k=10
    )

    for item in reranked_results:
        reranked_rows.append({
            "question_id": q_id,
            "question": question,
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "chunk": item["chunk"],
            "metadata": item["metadata"],
            "rerank_score": item["rerank_score"]
        })

reranked_df = pd.DataFrame(reranked_rows)

display(reranked_df)

,question_id,question,rank,chunk_id,chunk,metadata,rerank_score
0,1,What are the expenses covered under inpatient ...,1,chunk_506,Expenses: (i) Pre-Hospitalization Medical Expe...,"{'page': 2, 'document_type': 'Prospectus / Sal...",4.882286
1,1,What are the expenses covered under inpatient ...,2,chunk_851,for expenses incurred under Benefit 'In-patien...,"{'page': 1, 'source': 'care-supreme----policy-...",4.770754
2,1,What are the expenses covered under inpatient ...,3,chunk_575,"under this Benefit. However, these expenses ar...",{'document_type': 'Prospectus / Sales Literatu...,3.241331
3,1,What are the expenses covered under inpatient ...,4,chunk_705,Sum Insured (SI) – on annual basis (in Rs.) 3 ...,{'source': 'care-plus--(health-insurance-produ...,1.764421
4,1,What are the expenses covered under inpatient ...,5,chunk_502,"care due to Illness or Injury , which should b...",{'document_type': 'Prospectus / Sales Literatu...,1.218718
5,1,What are the expenses covered under inpatient ...,6,chunk_13,A mandatory Co-Payment of 10% per claim is app...,"{'page': 3, 'document_type': 'Policy Usage Gui...",1.100948
6,1,What are the expenses covered under inpatient ...,7,chunk_577,It all adds up - A trip to a hospital involves...,{'document_type': 'Prospectus / Sales Literatu...,-0.002956
7,1,What are the expenses covered under inpatient ...,8,chunk_574,is not available in case your age is greater t...,"{'page': 10, 'source': 'care-plus--(health-ins...",-0.157301
8,1,What are the expenses covered under inpatient ...,9,chunk_477,L) Pre-Hospitalization Medical Expenses and Po...,"{'document_type': 'Brochure', 'source': 'care-...",-0.232739
9,1,What are the expenses covered under inpatient ...,10,chunk_57,caesarean sections incurred during hospitaliza...,{'source': 'care-freedom-(health-insurance-pro...,-3.836710


In [37]:
reranked_metrics = evaluate_retriever(
    reranked_df,
    "Hybrid + Reranking"
)

display(reranked_metrics)

,Question,Method,K,Precision,Recall
0,Q1,Hybrid + Reranking,1,1.000,0.500
1,Q1,Hybrid + Reranking,3,0.333,0.500
2,Q1,Hybrid + Reranking,5,0.400,1.000
3,Q1,Hybrid + Reranking,10,0.200,1.000
4,Q2,Hybrid + Reranking,1,0.000,0.000
5,Q2,Hybrid + Reranking,3,0.667,0.500
6,Q2,Hybrid + Reranking,5,0.400,0.500
7,Q2,Hybrid + Reranking,10,0.300,0.750
8,Q3,Hybrid + Reranking,1,0.000,0.000
9,Q3,Hybrid + Reranking,3,0.333,0.333


In [38]:
all_metrics = pd.concat(
    [
        dense_metrics,
        bm25_metrics,
        hybrid_metrics,
        reranked_metrics
    ],
    ignore_index=True
)

final_comparison = (
    all_metrics
    .groupby(["Method", "K"])[["Precision", "Recall"]]
    .mean()
    .reset_index()
    .round(3)
)

display(final_comparison)

,Method,K,Precision,Recall
0,BM25,1,0.200,0.067
1,BM25,3,0.200,0.183
2,BM25,5,0.160,0.250
3,BM25,10,0.140,0.380
4,Dense,1,0.800,0.223
5,Dense,3,0.600,0.553
6,Dense,5,0.480,0.727
7,Dense,10,0.340,1.000
8,Hybrid,1,0.800,0.223
9,Hybrid,3,0.600,0.520


In [39]:
final_summary = (
    all_metrics
    .groupby(["Method", "K"])[["Precision", "Recall"]]
    .mean()
    .reset_index()
    .round(3)
)

display(final_summary)

,Method,K,Precision,Recall
0,BM25,1,0.200,0.067
1,BM25,3,0.200,0.183
2,BM25,5,0.160,0.250
3,BM25,10,0.140,0.380
4,Dense,1,0.800,0.223
5,Dense,3,0.600,0.553
6,Dense,5,0.480,0.727
7,Dense,10,0.340,1.000
8,Hybrid,1,0.800,0.223
9,Hybrid,3,0.600,0.520


In [40]:
final_summary.to_csv(
    "retrieval_evaluation_summary.csv",
    index=False
)

print("Evaluation summary saved successfully!")

Evaluation summary saved successfully!


In [41]:
all_metrics.to_csv(
    "retrieval_evaluation_detailed.csv",
    index=False
)

print("Detailed evaluation results saved successfully!")

Detailed evaluation results saved successfully!
